# 0. Imports

In [57]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from itertools import combinations
from collections import Counter
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
import re
from sklearn.metrics import (accuracy_score, mean_squared_error,
                             precision_recall_fscore_support)
from math import sqrt
import optuna
from optuna import Trial
import itertools

In [59]:
data_path = '/content/drive/MyDrive/UST/Year 5/Spring/COMP4332/Projects/Project 1/data'

# 1. Helper Functions

## 1.1 Find the most frequent combinations of categories in a df column

In [60]:
def get_top_k_p_combinations(df, comb_p, topk, output_freq=False):
    '''
    params:
        df: input dataframe
        comb_p: number of elements in each combination (e.g., there are two elements in the combination {fried chicken, chicken and waffle}, and three elements in the combination {fried chicken, chicken and waffle, chicken fried rice})
        topk: number of most frequent combinations to retrieve
        output_freq: whether to return the frequencies of retrieved combinations

    return:
        1. output_freq = True: a list X where each element is a tuple containing a combination tuple and corresponding frequency, and the elements are stored in the descending order of their frequencies
        2. output_freq = False: a list X where each element is a tuple containing a combination tuple, and the elements are stored in the descending order of their frequencies
    '''
    def get_category_combinations(categories, comb_p=2):
        return list(combinations(categories, comb_p))
    all_categories_p_combos = df["category"].apply(lambda x: get_category_combinations(x, comb_p)).values.tolist()
    all_categories_p_combos = [tuple(t) for item in all_categories_p_combos for t in item]
    tmp = dict(Counter(all_categories_p_combos))
    sorted_categories_combinations = list(sorted(tmp.items(), key=lambda x: x[1], reverse=True))
    if output_freq:
        return sorted_categories_combinations[:topk]
    else:
        return [t[0] for t in sorted_categories_combinations[:topk]]

### Sample Usage

In [61]:
# Sample data
data = [
    {"id": 1, "item": "Fried Chicken", "category": ["chicken", "fried", "southern"]},
    {"id": 2, "item": "Chicken and Waffles", "category": ["chicken", "breakfast", "southern"]},
    {"id": 3, "item": "Chicken Fried Rice", "category": ["chicken", "rice", "asian"]},
    {"id": 4, "item": "Beef Stir Fry", "category": ["beef", "asian", "rice"]},
    {"id": 5, "item": "Waffles", "category": ["breakfast", "sweet"]},
    {"id": 6, "item": "Southern BBQ", "category": ["southern", "bbq", "pork"]},
    {"id": 7, "item": "Chicken Curry", "category": ["chicken", "asian", "spicy"]}
]

df = pd.DataFrame(data)

# Get top 3 pairs of categories (comb_p=2)
top_pairs = get_top_k_p_combinations(df, comb_p=2, topk=3, output_freq=True)
print("Top 3 category pairs with frequencies:")
for pair, freq in top_pairs:
    print(f"{pair}: {freq}")

# Get top 3 triplets of categories (comb_p=3)
top_triplets = get_top_k_p_combinations(df, comb_p=3, topk=3, output_freq=False)
print("\nTop 3 category triplets:")
for triplet in top_triplets:
    print(triplet)

Top 3 category pairs with frequencies:
('chicken', 'southern'): 2
('chicken', 'asian'): 2
('chicken', 'fried'): 1

Top 3 category triplets:
('chicken', 'fried', 'southern')
('chicken', 'breakfast', 'southern')
('chicken', 'rice', 'asian')


## 1.2 Create a OHE (or "wide feature representation") of each row in the column of a df

In [62]:
def get_wide_features(df, selected_categories_to_idx, top_combinations):
    '''
    params:
        df: input dataframe
        selected_categories_to_idx: a dictionary mapping item categories to corrresponding integral indices
        top_combinations: a list containing retrieved mostly frequent combinantions of item categories

    return:
        a numpy array where each row contains the categorical features' binary encodings and cross product transformations for the corresponding row of the input dataframe
    '''
    def categories_to_binary_output(categories):
        binary_output = [0 for _ in range(len(selected_categories_to_idx))]
        for category in categories:  # Fixed: iterate through the current categories
            if category in selected_categories_to_idx:
                binary_output[selected_categories_to_idx[category]] = 1
            else:
                binary_output[0] = 1
        return binary_output

    def categories_cross_transformation(categories):
        current_category_set = set(categories)
        corss_transform_output = [0 for _ in range(len(top_combinations))]
        for k, comb_k in enumerate(top_combinations):
            # Convert the tuple to a set before using set operations
            if len(current_category_set & set(comb_k)) == len(comb_k):
                corss_transform_output[k] = 1
            else:
                corss_transform_output[k] = 0
        return corss_transform_output

    category_binary_features = np.array(df.category.apply(lambda x: categories_to_binary_output(x)).values.tolist())
    category_cross_transform_features = np.array(df.category.apply(lambda x: categories_cross_transformation(x)).values.tolist())
    return np.concatenate((category_binary_features, category_cross_transform_features), axis=1)

### Sample Usage

In [63]:
# First, get the top category combinations
top_combinations = get_top_k_p_combinations(df, comb_p=2, topk=3, output_freq=False)

# Create a mapping of categories to indices
all_categories = set()
for categories in df['category']:
    all_categories.update(categories)

selected_categories_to_idx = {cat: idx for idx, cat in enumerate(all_categories)}

# Generate wide features
wide_features = get_wide_features(df, selected_categories_to_idx, top_combinations)

print("Selected categories to indices:")
print(selected_categories_to_idx)
print("\nShape of wide features:", wide_features.shape)
print("\nWide features for first 2 items:")
print(wide_features[:2])

Selected categories to indices:
{'southern': 0, 'bbq': 1, 'fried': 2, 'spicy': 3, 'rice': 4, 'breakfast': 5, 'beef': 6, 'chicken': 7, 'sweet': 8, 'pork': 9, 'asian': 10}

Shape of wide features: (7, 14)

Wide features for first 2 items:
[[1 0 1 0 0 0 0 1 0 0 0 1 0 1]
 [1 0 0 0 0 1 0 1 0 0 0 1 0 0]]


# 2. Data Handling

## 2.1 Data Exploration

In [64]:
prediction_data = pd.read_csv(data_path + '/prediction.csv')
product_data = pd.read_json(data_path + '/product.json')
review_data = pd.read_csv(data_path + '/review.csv')
validation_data = pd.read_csv(data_path + '/validation.csv')

print(f"Prediction data shape: {prediction_data.shape}")
print(f"Product data shape: {product_data.shape}")
print(f"Review data shape: {review_data.shape}")
print(f"Validation data shape: {validation_data.shape}")

Prediction data shape: (6633, 3)
Product data shape: (6309, 17)
Review data shape: (52512, 5)
Validation data shape: (6596, 3)


In [65]:
prediction_data

,ReviewerID,ProductID,Star
0,A2MK1L1Y74WTWH,B01GT5XDFS,0
1,A19I68RW4PBT29,B00OME9OQQ,0
2,A1UPHTDW5GM12T,B01GSRNLOK,0
3,A1LFIFPYMOJ8RV,B01CUJYMR0,0
4,A10Y597K071WTQ,B004SI455Q,0
...,...,...,...
6628,A23Y4UGTFDMZOP,B00J5327X6,0
6629,A2PFNDDKHOOMZU,B01G0GIXJ2,0
6630,A1K4S4MWXI9E9M,B01FKDKB96,0
6631,AOLHNMI8G8R6K,B00NUDPR66,0


In [66]:
product_data

,category,tech1,description,fit,title,tech2,brand,feature,rank,details,main_cat,similar_item,date,price,imageURL,imageURLHighRes,ProductID
0,"[Kindle Store, Kindle eBooks, Biographies & Me...",,[],,,,Visit Amazon's Frank W. Abagnale Page,[],"59,404 Paid in Kindle Store (","{'File Size:': '1466 KB', 'Print Length:': '30...",Buy a Kindle,,NaT,,[],[],B000FBFMHU
1,"[Kindle Store, Kindle eBooks, Politics & Socia...",,[],,,,Visit Amazon's Karl Marx Page,[],"1,358,073 Paid in Kindle Store (","{'File Size:': '142 KB', 'Print Length:': '160...",Buy a Kindle,,NaT,,[],[],B000FC27TA
2,"[Kindle Store, Kindle eBooks, Romance]",,[],,,,Visit Amazon's Allison Brennan Page,[],"94,006 Paid in Kindle Store (","{'File Size:': '739 KB', 'Print Length:': '416...",Buy a Kindle,,NaT,,[],[],B000FCKPG2
3,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",,[],,,,Visit Amazon's Lynsay Sands Page,[],"31,652 Paid in Kindle Store (","{'File Size:': '1011 KB', 'Print Length:': '38...",Buy a Kindle,,NaT,,[],[],B000GCFWXW
4,"[Kindle Store, Kindle eBooks, Romance]",,[],,,,Visit Amazon's Fern Michaels Page,[],"1,031,468 Paid in Kindle Store (","{'File Size:': '519 KB', 'Print Length:': '320...",Buy a Kindle,,NaT,,[],[],B000JMKRTI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6304,"[Kindle Store, Kindle eBooks, Romance]",,[],,SCARS - Kindle edition,,Visit Amazon's Jaimie Roberts Page,[],"17,812 Paid in Kindle Store (","{'File Size:': '2096 KB', 'Print Length:': '42...",Buy a Kindle,,NaT,,[],[],B01HFFPC2I
6305,"[Kindle Store, Kindle eBooks, Romance]",,[],,Bride Of The Dragon - Kindle edition,,Visit Amazon's Georgette St. Clair Page,[],"220,668 Paid in Kindle Store (","{'File Size:': '4037 KB', 'Print Length:': '17...",Buy a Kindle,,NaT,,[],[],B01HFGNGYI
6306,"[Kindle Store, Kindle eBooks, Literature & Fic...",,[],,Miles (Special Forces Book 3) eBook,,Visit Amazon's KB Winters Page,[],"412,334 Paid in Kindle Store (","{'File Size:': '2444 KB', 'Print Length:': '29...",Buy a Kindle,,NaT,,[],[],B01HFUF1GK
6307,"[Kindle Store, Kindle eBooks, Romance]",,[],,The Billionaire&#39;s Triplets: Book One - Kin...,,Visit Amazon's Mia Caldwell Page,[],"9,672 Free in Kindle Store (","{'File Size:': '5487 KB', 'Print Length:': '27...",Buy a Kindle,,NaT,,[],[],B01HFTVMXM


In [67]:
review_data

,ReviewerID,ProductID,Text,Summary,Star
0,A1XJXYKOWCH9XT,B000FBFMHU,Liked the movie. Loved the book. It really giv...,Liked the movie. Loved the book!,5.0
1,A1K4S4MWXI9E9M,B000FC27TA,Purchased more out of curiosity than any real ...,"Not my favorite, but...",3.0
2,A3LF914GG87TWP,B000FC27TA,"I actually received this text as an ebook, sin...",An interesting read,4.0
3,A1CNQTCRQ35IMM,B000FCKPG2,REVIEWER'S OPINION:\nThis was labeled as roman...,This was labeled romance but there was less ro...,2.0
4,AU510CVD9XDG,B000GCFWXW,I have been saving the Argeneau novels for awh...,Science Fiction not Paranormal Romance,2.0
...,...,...,...,...,...
52507,A3JVZY05VLMYEM,B01FLJUZ0E,She can't do anything right according to her f...,What Can She Do,5.0
52508,A2U06P692IZOSF,B01FLJUZ0E,Better late than never!!\nKitty Konstantine ha...,BART & KITTY CAT MAKE SPARKS FLY!!,5.0
52509,A3RPL8JIS2XMJ3,B01FLJUZ0E,This book was great. Bartholomew finally gets ...,LOVE THE SAINTS,5.0
52510,A1XMFCMIANCQRW,B01FPYJS1M,I read for a honest review for the author.\nTh...,"Loved Lee and Raina together, Ricky is evil an...",4.0


In [68]:
validation_data

,ReviewerID,ProductID,Star
0,A25X28UZCW2J6G,B00K9V6B94,4.0
1,A1FUH1O6FCTUYG,B00GZANS6M,5.0
2,AAUVEEG5YLZAX,B01864DDVO,5.0
3,A3VQLGTYTL5196,B001BXNQ2O,5.0
4,A10JAUCIGVRW9F,B0116MZUS2,5.0
...,...,...,...
6591,A3TC60MGLW1I76,B00EHSUFD8,4.0
6592,AGE0YGLF7L2ZL,B014LQ18CW,4.0
6593,AT2ZB20OCU7X2,B00ZRDPPU0,4.0
6594,A1ACUN6A2LYVMW,B01EKIELGG,1.0


## 2.2 Preprocessing

### 2.2.1 Merging the train_df and val_df

In [69]:
# There are 184 rows in review_data for ProductIDs that do not exist in product_data. They have been removed through the merge.
train_df = pd.merge(review_data[['ReviewerID', 'ProductID', 'Star']], product_data, on='ProductID').reset_index(drop=True)
val_df = pd.merge(validation_data, product_data, on='ProductID').reset_index(drop=True)
train_df = train_df[['ReviewerID', 'ProductID', 'Star', 'category', 'brand', 'rank']]
val_df = val_df[['ReviewerID', 'ProductID', 'Star', 'category', 'brand', 'rank']]

In [70]:
# brand col --> remove "Visit Amazon's " and " Page"
train_df.loc[:, 'brand'] = train_df['brand'] \
    .str.replace("Visit Amazon's ", "") \
    .str.replace(" Page", "")
val_df.loc[:, 'brand'] = val_df['brand'] \
    .str.replace("Visit Amazon's ", "") \
    .str.replace(" Page", "")

In [71]:
# Extract numeric part using regex
train_df['rank'] = train_df['rank'].str.extract(r'^(\d+[,\d]*)')
train_df['rank'] = pd.to_numeric(train_df['rank'].str.replace(',', ''), errors='coerce')
val_df['rank'] = val_df['rank'].str.extract(r'^(\d+[,\d]*)')
val_df['rank'] = pd.to_numeric(val_df['rank'].str.replace(',', ''), errors='coerce')

# Fill missing values with the mean
train_df['rank'] = train_df['rank'].fillna(train_df['rank'].mean())
val_df['rank'] = val_df['rank'].fillna(train_df['rank'].mean())

# Convert to int
train_df['rank'] = train_df['rank'].astype(int)
val_df['rank'] = val_df['rank'].astype(int)

In [72]:
train_df

,ReviewerID,ProductID,Star,category,brand,rank
0,A1XJXYKOWCH9XT,B000FBFMHU,5.0,"[Kindle Store, Kindle eBooks, Biographies & Me...",Frank W. Abagnale,59404
1,A1K4S4MWXI9E9M,B000FC27TA,3.0,"[Kindle Store, Kindle eBooks, Politics & Socia...",Karl Marx,1358073
2,A3LF914GG87TWP,B000FC27TA,4.0,"[Kindle Store, Kindle eBooks, Politics & Socia...",Karl Marx,1358073
3,A1CNQTCRQ35IMM,B000FCKPG2,2.0,"[Kindle Store, Kindle eBooks, Romance]",Allison Brennan,94006
4,AU510CVD9XDG,B000GCFWXW,2.0,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",Lynsay Sands,31652
...,...,...,...,...,...,...
52323,A3JVZY05VLMYEM,B01FLJUZ0E,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Becca Fanning,759249
52324,A2U06P692IZOSF,B01FLJUZ0E,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Becca Fanning,759249
52325,A3RPL8JIS2XMJ3,B01FLJUZ0E,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Becca Fanning,759249
52326,A1XMFCMIANCQRW,B01FPYJS1M,4.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Marci Fawn,357833


In [73]:
val_df

,ReviewerID,ProductID,Star,category,brand,rank
0,A25X28UZCW2J6G,B00K9V6B94,4.0,"[Kindle Store, Kindle eBooks, Science Fiction ...",Erin Kellison,607901
1,A1FUH1O6FCTUYG,B00GZANS6M,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Evangeline Anderson,74267
2,AAUVEEG5YLZAX,B01864DDVO,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Lucia Jordan,13716
3,A3VQLGTYTL5196,B001BXNQ2O,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Eve Vaughn,730348
4,A30WJKGX1XG19Q,B00I4KRGPA,2.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Violet Duke,11765
...,...,...,...,...,...,...
5437,A3TC60MGLW1I76,B00EHSUFD8,4.0,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",Jaden Skye,485348
5438,AGE0YGLF7L2ZL,B014LQ18CW,4.0,"[Kindle Store, Kindle eBooks, Romance]",Susan Hatler,634245
5439,AT2ZB20OCU7X2,B00ZRDPPU0,4.0,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",Lee Child,285543
5440,A1ACUN6A2LYVMW,B01EKIELGG,1.0,"[Kindle Store, Kindle eBooks, Romance]",Lee Savino,16338


In [74]:
print(train_df.info(), '\n')
print(val_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52328 entries, 0 to 52327
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ReviewerID  52328 non-null  object 
 1   ProductID   52328 non-null  object 
 2   Star        52328 non-null  float64
 3   category    52328 non-null  object 
 4   brand       52328 non-null  object 
 5   rank        52328 non-null  int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 2.4+ MB
None 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5442 entries, 0 to 5441
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ReviewerID  5442 non-null   object 
 1   ProductID   5442 non-null   object 
 2   Star        5442 non-null   float64
 3   category    5442 non-null   object 
 4   brand       5442 non-null   object 
 5   rank        5442 non-null   int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 255.2+ 

### 2.2.2 Preparing continuous features

In [76]:
scaler = StandardScaler()
train_continuous_features = scaler.fit_transform(train_df[['rank']])
val_continuous_features = scaler.fit_transform(val_df[['rank']])

### 2.2.3 Preparing deep categorical features

In [91]:
deep_columns = ['brand']

deep_vocab_lens = []
for col_name in deep_columns:
    # Get unique values of this col
    unique_values = train_df[col_name].unique()

    # Put each unique value into a dict: (value, num), start num range from 1 onwards
    vocab = dict(zip(unique_values, range(1, len(unique_values) + 1)))

    # Add the number of items in the dict (+1 for unknown)
    deep_vocab_lens.append(len(vocab) + 1)

    # Add columns to the train_df to display the idx number of the OHE
    train_df[col_name + '_idx'] = train_df[col_name].apply(lambda x: vocab[x])

# Map ProductID to the _idx columns in train_df
deep_idx_columns = [t + "_idx" for t in deep_columns]
deep_categorical_features = dict(zip(train_df['ProductID'].values, train_df[deep_idx_columns].values.tolist()))

# Convert the _idx columns into np arrays for the model
train_deep_categorical_features = np.array(train_df['ProductID'].apply(lambda x: deep_categorical_features[x]).values.tolist())
val_deep_categorical_features = np.array(val_df['ProductID'].apply(lambda x: deep_categorical_features[x]).values.tolist())

In [92]:
len(train_deep_categorical_features), train_deep_categorical_features

(52328,
 array([[   1,    1,    1],
        [   3,    2,    2],
        [   3,    2,    2],
        ...,
        [2099, 6154, 3530],
        [ 882, 6171, 3555],
        [ 856, 6237, 3530]]))

In [93]:
len(val_deep_categorical_features), val_deep_categorical_features

(5442,
 array([[2033, 3041, 2133],
        [1028, 2379,   49],
        [ 868, 5513,  806],
        ...,
        [  19, 4870,  266],
        [1215, 6086, 3524],
        [1244, 5374, 3264]]))

### 2.2.4 Preparing wide features

Preparing binary encoding for each selected category

In [80]:
# Collect all categories from the category column
all_categories = []
for category_list in train_df.category.values:
    # Since category is already a list, we don't need to split it
    for category in category_list:
        all_categories.append(category)

# Sort all unique values of the categories by their frequencies in descending order
from collections import Counter
category_sorted = sorted(Counter(all_categories).items(), key=lambda x: x[1], reverse=True)

# Select top 500 most frequent categories
selected_categories = [t[0] for t in category_sorted[:500]]

# Create a dictionary mapping each selected category to a unique integral index
selected_categories_to_idx = dict(zip(selected_categories, range(1, len(selected_categories) + 1)))

# Map all categories unseen in the df to index 0
selected_categories_to_idx['unk'] = 0

# Create a dictionary mapping each integral index to corresponding category
idx_to_selected_categories = {val: key for key, val in selected_categories_to_idx.items()}

In [81]:
idx_to_selected_categories

{1: 'Kindle Store',
 2: 'Kindle eBooks',
 3: 'Literature & Fiction',
 4: 'Romance',
 5: 'Mystery, Thriller & Suspense',
 6: 'Science Fiction & Fantasy',
 7: 'Religion & Spirituality',
 8: 'Teen & Young Adult',
 9: "Children's eBooks",
 10: 'Health, Fitness & Dieting',
 11: 'Business & Money',
 12: 'Humor & Entertainment',
 13: 'Cookbooks, Food & Wine',
 14: '</span>',
 15: 'Biographies & Memoirs',
 16: 'Kindle Keyboard',
 17: 'Kindle DX',
 18: 'Kindle (2nd Generation)',
 19: 'Kindle (5th Generation)',
 20: 'Activities, Puzzles & Games',
 21: 'Politics & Social Sciences',
 22: 'Arts & Photography',
 23: 'Literature &amp; Fiction',
 24: 'Reference',
 25: 'Kindle Paperwhite',
 26: 'Kindle Paperwhite (5th Generation)',
 27: 'Kindle Touch',
 28: 'Crafts, Hobbies & Home',
 29: 'History',
 30: 'Computers & Technology',
 31: 'Science & Math',
 32: 'Self-Help',
 33: 'Education & Teaching',
 34: 'Word Games',
 35: 'Parenting & Relationships',
 36: 'Comics, Manga & Graphic Novels',
 37: 'Travel',

Preparing cross product transformation for categories

In [82]:
# Get most frequent categories combinantions using the utility function defined previously and store them in the folloing list
top_combinations = []

# Get top 50 most frequent two-categories combinantions in the train set
top_combinations += get_top_k_p_combinations(train_df, 2, 50, output_freq=False)

# Get top 30 most frequent three-categories combinantions in the train set
top_combinations += get_top_k_p_combinations(train_df, 3, 30, output_freq=False)

# Get top 20 most frequent four-categories combinantions in the train set
top_combinations += get_top_k_p_combinations(train_df, 4, 20, output_freq=False)

# Convert each combinantion in the list to a set data structure
top_combinations = [set(t) for t in top_combinations]

In [83]:
top_combinations

[{'Kindle Store', 'Kindle eBooks'},
 {'Kindle Store', 'Literature & Fiction'},
 {'Kindle eBooks', 'Literature & Fiction'},
 {'Kindle Store', 'Romance'},
 {'Kindle eBooks', 'Romance'},
 {'Kindle Store', 'Mystery, Thriller & Suspense'},
 {'Kindle eBooks', 'Mystery, Thriller & Suspense'},
 {'Kindle Store', 'Science Fiction & Fantasy'},
 {'Kindle eBooks', 'Science Fiction & Fantasy'},
 {'Kindle Store', 'Religion & Spirituality'},
 {'Kindle eBooks', 'Religion & Spirituality'},
 {'Kindle Store', 'Teen & Young Adult'},
 {'Kindle eBooks', 'Teen & Young Adult'},
 {"Children's eBooks", 'Kindle Store'},
 {"Children's eBooks", 'Kindle eBooks'},
 {'Health, Fitness & Dieting', 'Kindle Store'},
 {'Health, Fitness & Dieting', 'Kindle eBooks'},
 {'Business & Money', 'Kindle Store'},
 {'Business & Money', 'Kindle eBooks'},
 {'Humor & Entertainment', 'Kindle Store'},
 {'Humor & Entertainment', 'Kindle eBooks'},
 {'Cookbooks, Food & Wine', 'Kindle Store'},
 {'Cookbooks, Food & Wine', 'Kindle eBooks'},
 {'

In [84]:
train_wide_features = get_wide_features(train_df, selected_categories_to_idx, top_combinations)
val_wide_features = get_wide_features(val_df, selected_categories_to_idx, top_combinations)

In [85]:
train_wide_features

array([[0, 1, 1, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0],
       ...,
       [0, 1, 1, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0]])

In [86]:
val_wide_features

array([[0, 1, 1, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0],
       ...,
       [0, 1, 1, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0]])

### 2.2.5 Concatenating deep categorical features and wide features as an input list

In [87]:
train_features = []
train_features.append(train_continuous_features)
train_features += [train_deep_categorical_features[:, i] for i in range(train_deep_categorical_features.shape[1])]
train_features.append(train_wide_features)

val_features = []
val_features.append(val_continuous_features)
val_features += [val_deep_categorical_features[:, i] for i in range(val_deep_categorical_features.shape[1])]
val_features.append(val_wide_features)

In [88]:
train_features

[array([[-0.58201175],
        [ 3.32779989],
        [ 3.32779989],
        ...,
        [ 1.52496249],
        [ 0.3164475 ],
        [ 1.12077941]]),
 array([   1,    2,    2, ..., 3530, 3555, 3530]),
 array([[0, 1, 1, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0],
        ...,
        [0, 1, 1, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0]])]

In [89]:
val_features

[array([[ 1.65026486],
        [-0.41399257],
        [-0.64822213],
        ...,
        [ 0.40328678],
        [-0.63807944],
        [ 0.8657152 ]]),
 array([2133,   49,  806, ...,  266, 3524, 3264]),
 array([[0, 1, 1, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0],
        ...,
        [0, 1, 1, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0]])]

# 3. Model Implementation

## 3.1 Set up Dataset and Model

In [33]:
class RatingDataset(Dataset):
    def __init__(self, features, ratings):
        self.continuous = torch.FloatTensor(features[0])
        self.deep_categorical = [torch.LongTensor(f) for f in features[1:-1]]
        self.wide = torch.FloatTensor(features[-1])
        self.ratings = torch.FloatTensor(ratings)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.continuous[idx],
            *[dc[idx] for dc in self.deep_categorical],
            self.wide[idx],
            self.ratings[idx]
        )

In [34]:
class WideDeepModel(nn.Module):
    def __init__(self, len_continuous, deep_vocab_lens, len_wide, embed_size):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_size, embed_size) for vocab_size in deep_vocab_lens
        ])

        self.dnn = nn.Sequential(
            nn.Linear(len_continuous + len(deep_vocab_lens) * embed_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.fc = nn.Linear(64 + len_wide, 1)

    def forward(self, continuous, *args):
        deep_categorical = args[:len(self.embeddings)]
        wide = args[-1]

        embeds = []
        for i, emb_layer in enumerate(self.embeddings):
            embeds.append(emb_layer(deep_categorical[i]))
        embeds = torch.cat(embeds, 1)

        deep_input = torch.cat([continuous, embeds], dim=1)
        dnn_output = self.dnn(deep_input)

        combined = torch.cat([dnn_output, wide], dim=1)
        return self.fc(combined).squeeze()


In [35]:
train_ratings = train_df['Star'].values
val_ratings = val_df['Star'].values

## 3.2 Training the model

In [36]:
train_dataset = RatingDataset(train_features, train_ratings)
val_dataset = RatingDataset(val_features, val_ratings)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device", device)

model = WideDeepModel(
    len_continuous=1,
    deep_vocab_lens=deep_vocab_lens,
    len_wide=train_wide_features.shape[1],
    embed_size=100
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 3
train_losses = []
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for continuous, brand, wide, ratings in train_loader:
        optimizer.zero_grad()
        outputs = model(
            continuous.to(device),
            brand.to(device),
            wide.to(device)
        )
        loss = criterion(outputs, ratings.to(device))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    train_losses.append(epoch_loss / len(train_loader))
    print(f"Epoch {epoch+1} loss: {train_losses[-1]:.4f}")

using device cuda
Epoch 1 loss: 1.0991
Epoch 2 loss: 0.8548
Epoch 3 loss: 0.8424


## 3.3 Evaluation on validation data

In [37]:
def rmse(pred, actual):
    # Ignore nonzero terms.
    pred = pred[actual.nonzero()].flatten()
    actual = actual[actual.nonzero()].flatten()
    return sqrt(mean_squared_error(pred, actual))

In [38]:
val_reviewer_ids = val_df["ReviewerID"].values
val_product_ids = val_df["ProductID"].values

model.eval()
predictions = []
all_reviewer_ids = []
all_product_ids = []
index_offset = 0

with torch.no_grad():
    for batch_idx, (continuous, brand, wide, ratings) in enumerate(val_loader):
        continuous = continuous.to(device)
        brand = brand.to(device)
        wide = wide.to(device)

        outputs = model(continuous, brand, wide)

        current_preds = outputs.cpu().numpy().flatten().tolist()
        predictions.extend(current_preds)

        batch_size = len(continuous)
        batch_ids = val_reviewer_ids[index_offset:index_offset+batch_size]
        batch_prods = val_product_ids[index_offset:index_offset+batch_size]
        all_reviewer_ids.extend(batch_ids)
        all_product_ids.extend(batch_prods)

        index_offset += batch_size

pred_df = pd.DataFrame({
    "ReviewerID": all_reviewer_ids,
    "ProductID": all_product_ids,
    "Star": predictions
})

df = pd.merge(
    val_df,
    pred_df,
    how="left",
    on=["ReviewerID", "ProductID"]
)

print("VALIDATION RMSE: ", rmse(df["Star_y"].values, df["Star_x"].values))

VALIDATION RMSE:  1.0377266152675895


## 3.4 Hyperparameter Tuning

In [39]:
class RatingDataset(Dataset):
    def __init__(self, features, ratings):
        self.continuous = torch.FloatTensor(features[0])
        self.deep_categorical = [torch.LongTensor(f) for f in features[1:-1]]
        self.wide = torch.FloatTensor(features[-1])
        self.ratings = torch.FloatTensor(ratings)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.continuous[idx],
            *[dc[idx] for dc in self.deep_categorical],
            self.wide[idx],
            self.ratings[idx]
        )

class WideDeepModel(nn.Module):
    def __init__(self, len_continuous, deep_vocab_lens, len_wide, embed_size, hidden_dims, dropout):
        super().__init__()
        # Embeddings
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_size, embed_size) for vocab_size in deep_vocab_lens
        ])

        layers = []
        input_dim = len_continuous + len(deep_vocab_lens) * embed_size
        for dim in hidden_dims:
            layers.append(nn.Linear(input_dim, dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            input_dim = dim
        self.dnn = nn.Sequential(*layers)
        self.fc = nn.Linear(hidden_dims[-1] + len_wide, 1)

    def forward(self, continuous, *args):
        deep_categorical = args[:len(self.embeddings)]
        wide = args[-1]

        embeds = []
        for i, emb_layer in enumerate(self.embeddings):
            embeds.append(emb_layer(deep_categorical[i]))
        embeds = torch.cat(embeds, 1)

        deep_input = torch.cat([continuous, embeds], dim=1)
        dnn_output = self.dnn(deep_input)

        combined = torch.cat([dnn_output, wide], dim=1)
        return self.fc(combined).squeeze()

In [40]:
def rmse(pred, actual):
    # Ignore nonzero terms.
    pred = pred[actual.nonzero()].flatten()
    actual = actual[actual.nonzero()].flatten()
    return sqrt(mean_squared_error(pred, actual))

In [42]:
def train_one_epoch(model, device, train_loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for batch_idx, batch_data in enumerate(train_loader):
        continuous = batch_data[0].to(device)
        deep_cats = [bd.to(device) for bd in batch_data[1:-2]]
        wide = batch_data[-2].to(device)
        ratings = batch_data[-1].to(device)

        optimizer.zero_grad()

        # Forward
        outputs = model(continuous, *deep_cats, wide)
        loss = criterion(outputs, ratings)

        # Backward
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    return avg_loss


def evaluate_rmse(model, device, val_loader):
    model.eval()
    preds = []
    actuals = []
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(val_loader):
            continuous = batch_data[0].to(device)
            deep_cats = [bd.to(device) for bd in batch_data[1:-2]]
            wide = batch_data[-2].to(device)
            ratings = batch_data[-1].to(device)

            outputs = model(continuous, *deep_cats, wide)

            preds.append(outputs.cpu())
            actuals.append(ratings.cpu())

    # Concatenate all predictions and actuals
    preds = torch.cat(preds).numpy()
    actuals = torch.cat(actuals).numpy()
    return rmse(preds, actuals)

In [50]:
def hyperparameter_search(
    train_loader,
    val_loader,
    len_continuous,
    deep_vocab_lens,
    len_wide,
    device
):
    embed_sizes = [8, 16]
    hidden_dims_list = [
        [256, 128, 64],
        [128, 64]
    ]
    dropouts = [0.2, 0.3]
    learning_rates = [1e-3, 1e-4]

    best_rmse = float("inf")
    best_params = None
    criterion = nn.MSELoss()

    for embed_size, hidden_dims, dropout, lr in itertools.product(
        embed_sizes, hidden_dims_list, dropouts, learning_rates
    ):
        print("="*60)
        print(f"Starting new run with:")
        print(f"  embed_size={embed_size}, hidden_dims={hidden_dims}, dropout={dropout}, lr={lr}")

        model = WideDeepModel(
            len_continuous=len_continuous,
            deep_vocab_lens=deep_vocab_lens,
            len_wide=len_wide,
            embed_size=embed_size,
            hidden_dims=hidden_dims,
            dropout=dropout
        ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=lr)

        n_epochs = 10
        for epoch in range(n_epochs):
            train_loss = train_one_epoch(model, device, train_loader, optimizer, criterion)

            val_rmse = evaluate_rmse(model, device, val_loader)

            print(
                f"Epoch {epoch+1}/{n_epochs}: "
                f"Train Loss = {train_loss:.4f}, "
                f"Validation RMSE = {val_rmse:.4f}"
            )

        current_rmse = evaluate_rmse(model, device, val_loader)

        if current_rmse < best_rmse:
            best_rmse = current_rmse
            best_params = {
                "embed_size": embed_size,
                "hidden_dims": hidden_dims,
                "dropout": dropout,
                "learning_rate": lr,
                "rmse": best_rmse
            }

        print(
            f"Finished run: embed={embed_size}, hidden={hidden_dims}, "
            f"dropout={dropout}, lr={lr} => Validation RMSE: {current_rmse:.4f}"
        )

    print("\nBest hyperparameters found:")
    print(best_params)
    return best_params

In [51]:
train_loader = DataLoader(train_dataset, batch_size=32)
val_loader   = DataLoader(val_dataset, batch_size=32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_params = hyperparameter_search(
    train_loader,
    val_loader,
    len_continuous=1,
    deep_vocab_lens=deep_vocab_lens,
    len_wide=train_wide_features.shape[1],
    device=device
)

Starting new run with:
  embed_size=8, hidden_dims=[256, 128, 64], dropout=0.2, lr=0.001
Epoch 1/10: Train Loss = 1.3297, Validation RMSE = 1.0381
Epoch 2/10: Train Loss = 0.9489, Validation RMSE = 1.0365
Epoch 3/10: Train Loss = 0.8924, Validation RMSE = 1.0379
Epoch 4/10: Train Loss = 0.8711, Validation RMSE = 1.0305
Epoch 5/10: Train Loss = 0.8641, Validation RMSE = 1.0302
Epoch 6/10: Train Loss = 0.8579, Validation RMSE = 1.0296
Epoch 7/10: Train Loss = 0.8509, Validation RMSE = 1.0293
Epoch 8/10: Train Loss = 0.8431, Validation RMSE = 1.0295
Epoch 9/10: Train Loss = 0.8351, Validation RMSE = 1.0300
Epoch 10/10: Train Loss = 0.8278, Validation RMSE = 1.0299
Finished run: embed=8, hidden=[256, 128, 64], dropout=0.2, lr=0.001 => Validation RMSE: 1.0299
Starting new run with:
  embed_size=8, hidden_dims=[256, 128, 64], dropout=0.2, lr=0.0001
Epoch 1/10: Train Loss = 2.5188, Validation RMSE = 1.1721
Epoch 2/10: Train Loss = 1.2804, Validation RMSE = 1.0604
Epoch 3/10: Train Loss = 1.11